# 01b: 特徴量エンジン検証

実データで特徴量エンジン(`src/features/`)を動作させ、各特徴量の品質を確認する。

In [ ]:
import sys
sys.path.insert(0, ".")
sys.path.insert(0, "src")

%run "./00_setup.ipynb"

In [ ]:
from db.connection import DatabaseConnection
from features.feature_engine import FeatureEngine
from features.leakage_validators import validate_no_future_leakage
import pandas as pd

conn = DatabaseConnection()
engine = conn.get_engine()

# データロード (2015年以降)
print("データをロード中...")
race_df = conn.load_races("20150101", "20261231")
entry_df = conn.load_entries_with_results("20150101", "20261231")
odds_df = conn.load_odds_snapshots("20150101", "20261231")

# 時系列オッズは大きいため直近3年分のみ
odds_ts_df = conn.load_odds_time_series_range("20220101", "20261231")

print(f"  races: {len(race_df):,}")
print(f"  entries: {len(entry_df):,}")
print(f"  odds_snapshots: {len(odds_df):,}")
print(f"  odds_timeseries: {len(odds_ts_df):,}")

In [ ]:
print("特徴量を生成中...")
feat_engine = FeatureEngine()
feat_df = feat_engine.build_all(race_df, entry_df, odds_df, odds_ts_df=odds_ts_df)
print(f"  生成データ: {len(feat_df):,} 行 × {len(feat_df.columns)} 列")

In [ ]:
print("=== 欠損値チェック ===")
nulls = feat_df.isnull().sum()
null_cols = nulls[nulls > 0].sort_values(ascending=False)
if len(null_cols) == 0:
    print("  欠損値なし ✓")
else:
    for col, cnt in null_cols.items():
        pct = cnt / len(feat_df) * 100
        print(f"  {col:40s} {cnt:>8,} ({pct:.1f}%)")

In [ ]:
import numpy as np
print("=== Inf値チェック ===")
numeric_cols = feat_df.select_dtypes(include=[np.number]).columns
inf_cols = []
for col in numeric_cols:
    if np.isinf(feat_df[col]).any():
        inf_cols.append((col, np.isinf(feat_df[col]).sum()))
if not inf_cols:
    print("  Inf値なし ✓")
else:
    for col, cnt in inf_cols:
        print(f"  {col}: {cnt} 件")

In [ ]:
print("=== 特徴量の基本統計 ===")
feature_cols = [c for c in feat_df.columns if c not in [
    "race_id", "race_date", "umaban", "ketto_num", "year", "month_day",
    "jyo_cd", "kaiji", "nichiji", "race_num", "surface", "surface_key"
]]
feat_df[feature_cols].describe().T

In [ ]:
print("=== 未来リーク検証 ===")
try:
    result = validate_no_future_leakage(feat_df)
    print(f"  結果: {result}")
except Exception as e:
    print(f"  エラー（データ未ロードの場合）: {e}")